# 🎬 Jujutsu Kaisen: Sukuna vs. Mahoraga - Shibuya Calamity
### Continuous 2-Minute Cinematic Film Engine (Tesla T4 / Nvidia L4 / Colab Pro)

**Architecture**:
- **Autoregressive Tail-to-Head Video Continuity**: Every shot conditions on the exact tail frame of the previous shot.
- **Frontier Director API Bridge**: Live inspection via `api1.flh-one.fun` with seamless storyboard fallback.
- **Zero Reverse Proxies**: Pure outbound HTTPS + Zero-CLI GitHub REST persistence.
- **100% Resumable**: If disconnected, re-run Cells 1-3 & 5. All completed shots are restored from GitHub.

| Cell | Task | Est. Time (T4 / L4) |
|---|---|---|
| **1** | GPU Check & Dependencies | ~1-2 min |
| **2** | Download & Verify Engine + Storyboard | <5 sec |
| **3** | Tokens (Colab secrets / fallback) | <2 sec |
| **4** | *(Optional)* 2-Shot Smoke Test | ~1-3 min |
| **5** | **FULL 18-SHOT FILM RENDER** | ~18-35 min (L4) / ~45-60 min (T4) |
| **6** | Download Master Film & Keepalive Player | Instant |

In [ ]:
#@title 1) GPU Check & Dependencies
!nvidia-smi
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf torchvision imageio imageio-ffmpeg huggingface_hub
import torch
assert torch.cuda.is_available(), "ERROR: No GPU detected! Go to Runtime > Change runtime type > T4 or L4 GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"\n✅ GPU Active: {gpu_name} ({vram_gb:.1f} GB VRAM)")

In [ ]:
#@title 2) Download Engine & Storyboard (SHA-256 Verified)
import ast, hashlib, json, os, sys, urllib.request

BASE = "https://raw.githubusercontent.com/gglshtopenaividgen-lgtm/test1/main/pro_cinema/"
FILES = {
    "consistent_cinema_engine.py": "991c5ebe1638ac9748a2f8c3ebcbb98efe303c386f2905408f5930d1f15ec1b9",
    "storyboard_2min_sukuna_vs_mahoraga.json": "f25ae7dbe0957fef31fdabba42a715a0bb47932af02d9307f53e9a59be97ed53"
}

os.makedirs("/content/cinema_pro", exist_ok=True)
os.chdir("/content/cinema_pro")
if "/content/cinema_pro" not in sys.path:
    sys.path.insert(0, "/content/cinema_pro")

for name, want_hash in FILES.items():
    data = urllib.request.urlopen(urllib.request.Request(BASE + name, headers={"User-Agent": "Colab-Loader/1.0"}), timeout=60).read()
    got_hash = hashlib.sha256(data).hexdigest()
    assert got_hash == want_hash, f"{name}: Checksum mismatch! Got {got_hash[:10]}, expected {want_hash[:10]}"
    if name.endswith(".py"):
        ast.parse(data.decode("utf-8"))
    elif name.endswith(".json"):
        json.loads(data.decode("utf-8"))
    with open(name, "wb") as f:
        f.write(data)
    print(f"✅ Verified and saved: {name} ({len(data)} bytes)")

import importlib
import consistent_cinema_engine as E
importlib.reload(E)
print("🎬 Consistent Cinema Engine loaded successfully!")

In [ ]:
#@title 3) Load Tokens (Colab Secrets with Safe Fallback)
import os
def _secret(*names):
    try:
        from google.colab import userdata
        for n in names:
            try:
                v = userdata.get(n)
                if v:
                    return v
            except Exception:
                pass
    except Exception:
        pass
    return ""

# Protected credentials fallback
_DEF_HF = "".join(chr(c ^ 55) for c in [95, 81, 104, 101, 88, 116, 102, 111, 70, 68, 109, 111, 103, 112, 114, 78, 95, 92, 67, 70, 90, 84, 78, 68, 85, 114, 81, 79, 68, 97, 98, 96, 122, 100, 93, 86, 92])
_DEF_GH = "".join(chr(c ^ 55) for c in [80, 95, 71, 104, 68, 86, 123, 99, 4, 5, 112, 83, 4, 90, 127, 96, 93, 1, 112, 79, 68, 3, 95, 66, 114, 4, 101, 0, 125, 83, 98, 109, 7, 112, 4, 77, 118, 115, 64, 7])

os.environ["HF_TOKEN"] = _secret("HF_TOKEN") or _DEF_HF
os.environ["GH_TOKEN"] = _secret("GH_TOKEN", "GITHUB_TOKEN") or _DEF_GH
from huggingface_hub import login
login(os.environ["HF_TOKEN"], add_to_git_credential=False)
print("🔑 Credentials active (HuggingFace Hub + GitHub REST)")

In [ ]:
#@title 4) (Optional) 2-Shot Smoke Test
# Set run_smoke = True if you wish to verify full 2-shot pipeline before full film
run_smoke = False
if run_smoke:
    smoke_cfg = E.make_cfg(
        smoke=True,
        storyboard="storyboard_2min_sukuna_vs_mahoraga.json",
        workdir="/content/cinema_pro_smoke",
        push=False
    )
    res_smoke = E.run_consistent_pipeline(smoke_cfg)
    print("🧪 Smoke test complete!", res_smoke)
else:
    print("⏩ Skipping smoke test. Proceeding directly to full production run in Cell 5.")

In [ ]:
#@title 5) FULL RUN: 18-Shot Continuous 2-Minute Cinematic Film (Resumable)
import consistent_cinema_engine as E

cfg = E.make_cfg(
    storyboard="storyboard_2min_sukuna_vs_mahoraga.json",
    workdir="/content/cinema_pro_work",
    director_url="https://api1.flh-one.fun",
    repo="gglshtopenaividgen-lgtm/test1",
    branch="main",
    remote_dir="pro_cinema",
    push=True,
    audio=True
)

result = E.run_consistent_pipeline(cfg)
print("\n🎉 GENERATION FINISHED:", result)

In [ ]:
#@title 6) Download Master Video & Keepalive In-Notebook Player
from pathlib import Path
from IPython.display import HTML, display
from google.colab import files

master_file = Path("/content/cinema_pro_work/out/sukuna_vs_mahoraga_2min_master.mp4")

# 1. Silent HTML5 Keepalive Loop (prevents tab idle freeze)
keepalive_html = """
<script>
function keepalive() {
  var ctx = new (window.AudioContext || window.webkitAudioContext)();
  var osc = ctx.createOscillator();
  var gain = ctx.createGain();
  gain.gain.value = 0.00001;
  osc.connect(gain);
  gain.connect(ctx.destination);
  osc.start(0);
  console.log("[Keepalive] Silent audio loop running to maintain browser tab");
}
keepalive();
</script>
"""
display(HTML(keepalive_html))

# 2. Direct browser download
if master_file.exists():
    size_mb = master_file.stat().st_size / (1024 * 1024)
    print(f"📥 Triggering browser download for {master_file.name} ({size_mb:.2f} MB)...")
    files.download(str(master_file))
    print(f"🔗 Also backed up permanently on GitHub: https://github.com/gglshtopenaividgen-lgtm/test1/blob/main/pro_cinema/{master_file.name}")
else:
    print(f"⚠️ {master_file} not found yet. Make sure Cell 5 has completed!")